# Boruta SHAP : a feature selection techniques

Having too many features in building ML model allows to some issues such as curse of dimensionality. That's why we employ feature selection techniques to try to remove useless features from our datasets. But, to realize this selection, we need to identify useful features. 

In this notebook, we use Boruta, a feature selection algorithm based on two principles : 
1. **shadow features** : evaluate the features importances in competing the features with randomized versions of these features, called *shadow features*, by using any ML model of preference. (*In this example, we use `RandomForest`*.)

2. **binomial distributions** : the outcomes of the ML model is binary, therefore we can perform the previous step several times (with various randomization) and build a binomial distribution out of the features. We insterested on the tails of the distribution and keep only the features on the *acceptance* area. 

In [8]:
# import library
import pandas as pd
import numpy as np
## from scikit-learn
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

## from boruta
from boruta import BorutaPy

## from SHAPley
from BorutaShap import BorutaShap

/Users/ranu/Documents/ML-Concept/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Import dataset
dataset = load_diabetes(as_frame = True)

In [10]:
# Gets the independent variables
X = dataset['data']
# Gets the dependent variable (the target)
y = dataset['target']
# Splits the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [11]:
# Defines the estimator used by the Boruta algorithm
estimator = RandomForestRegressor()

In [12]:
# Features selection

## with Boruta
## Creates the BorutaPy object
boruta = BorutaPy(estimator = estimator, n_estimators = 'auto', max_iter = 100)
## Fits Boruta
boruta.fit(np.array(X_train), np.array(y_train))
## Output
### Important features
important = list(X.columns[boruta.support_])
print(f"Features confirmed as important: {important}")
### Tentative features
tentative = list(X.columns[boruta.support_weak_])
print(f"Unconfirmed features (tentative): {tentative}")
### Unimportant features
unimportant = list(X.columns[~(boruta.support_ | boruta.support_weak_)])
print(f"Features confirmed as unimportant: {unimportant}")

## with BorutaShap
## Creates a BorutaShap selector for regression
selector = BorutaShap(importance_measure = 'shap', classification = False)
## Fits the selector
selector.fit(X = X_train, y = y_train, n_trials = 100, sample = False, verbose = True)

Features confirmed as important: ['bmi', 'bp', 's2', 's5']
Unconfirmed features (tentative): []
Features confirmed as unimportant: ['age', 'sex', 's1', 's3', 's4', 's6']


  0%|          | 0/100 [00:01<?, ?it/s]


AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.